In [ ]:
# =================================================================
# CELL 1 — setup and CSV picker
# =================================================================
%matplotlib widget
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit
from scipy.special import gamma as gamma_func
import glob
import ipywidgets as widgets
from IPython.display import display
 
 
def _weibull_comp(phi, lam, n):
    d     = 2.0 ** (-phi)
    ratio = d / lam
    norm  = (n ** (1.0 / n)) * gamma_func(1.0 + 1.0 / n)
    return (np.log(2.0) / norm) * ratio ** (n + 1.0) * np.exp(-ratio ** n / n)
 
 
def bi_weibull(phi, lam1, n1, lam2, n2, q):
    return q * _weibull_comp(phi, lam1, n1) + (1.0 - q) * _weibull_comp(phi, lam2, n2)
 
 
csv_files = sorted(glob.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError("No .csv files found in the current folder.")
 
csv_picker = widgets.Dropdown(
    options=csv_files,
    description='CSV file:',
    style={'description_width': 'initial'},
)
display(csv_picker)
 

In [ ]:
# =================================================================
# CELL 2 — load data using the file selected above
# =================================================================
DATASET_LABEL = csv_picker.value.rsplit('.', 1)[0]
print(f"Loading: {csv_picker.value}")
 
_raw = pd.read_csv(csv_picker.value, skiprows=1)
_raw = _raw.rename(columns={'Phi': 'phi', 'Mass (wt.%)': 'mass'})
_data = _raw[['phi', 'mass']].dropna(subset=['phi']).copy()
_data['mass'] = _data['mass'].fillna(0.0).astype(float)
_data['phi']  = _data['phi'].astype(float)
phi_labels = _data['phi'].to_numpy() - 0.5
mass_pct   = _data['mass'].to_numpy()
DENSITY = 2500.0  # kg/m3
 
mass_frac = mass_pct / mass_pct.sum()
 
print(f"Loaded {DATASET_LABEL}: {len(phi_labels)} points, "
      f"Phi range [{phi_labels.min():.2f}, {phi_labels.max():.2f}]")
 
 

In [ ]:
# =================================================================
# CELL 3 — interactive picker
# =================================================================
picked_points = []  # reset each time this cell runs
MAX_CLICKS = 2

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(phi_labels, mass_pct, width=0.90, color='#c0392b', alpha=0.55,
       edgecolor='grey', linewidth=0.5, label='Data')
ax.set_xlabel('Φ', fontsize=11)
ax.set_ylabel('Mass (wt.%)', fontsize=11)
ax.set_title('Click the peak of each mode (order doesn\'t matter)')
ax.set_xlim(phi_labels.min() - 1, phi_labels.max() + 1)
ax.set_ylim(bottom=0)
ax.grid(alpha=0.3)
ax.legend(fontsize=8)

colors = ['#2980b9', '#27ae60']


def onclick(event):
    if event.inaxes != ax or len(picked_points) >= MAX_CLICKS:
        return
    phi_click, y_click = event.xdata, event.ydata
    picked_points.append((phi_click, y_click))

    i = len(picked_points) - 1
    ax.axvline(phi_click, color=colors[i], linestyle='--', lw=1.5)
    ax.annotate(f'peak {i+1}\nΦ={phi_click:.2f}',
                xy=(phi_click, y_click), xytext=(5, 5),
                textcoords='offset points', color=colors[i], fontsize=9)
    fig.canvas.draw_idle()

    print(f"Peak {i+1} picked: Phi={phi_click:.3f}, wt%={y_click:.3f}")
    if len(picked_points) == MAX_CLICKS:
        print("Both peaks picked - run the next cell to fit.")


cid = fig.canvas.mpl_connect('button_press_event', onclick)
plt.tight_layout()
plt.show()

In [ ]:
# =================================================================
# CELL 4 — build p0 from clicks and fit
# =================================================================
if len(picked_points) < 2:
    raise RuntimeError(
        "Only got {} click(s) so far. Click twice on the Cell 2 plot "
        "(one per mode peak) before running this cell."
        .format(len(picked_points))
    )

# Sort by phi so pop 1 = coarse (smaller phi -> larger lambda),
# pop 2 = fine (larger phi -> smaller lambda)
(phi1, y1), (phi2, y2) = sorted(picked_points, key=lambda pt: pt[0])

# lambda has units of mm (same as diameter), so convert phi -> mm
# the same way the model does: d = 2^-phi
lam1_guess = 2.0 ** (-phi1)  # coarse -> larger lambda
lam2_guess = 2.0 ** (-phi2)  # fine   -> smaller lambda

# n (shape) isn't something you can read off a histogram by eye;
# default to a moderate value, same as the non-interactive script.
n1_guess = 1.0
n2_guess = 1.0

# q (coarse weight) from relative click heights
q_guess = y1 / (y1 + y2) if (y1 + y2) > 0 else 0.5

p0_w = [lam1_guess, n1_guess, lam2_guess, n2_guess, q_guess]
lb_w = [1e-4, 0.05, 1e-7, 0.05, 0.01]
ub_w = [500., 5.0, 10.,  5.0,  0.99]
p0_w = list(np.clip(p0_w, lb_w, ub_w))

print(f"Initial guess from clicks: lambda1={p0_w[0]:.4f} mm, n1={p0_w[1]:.3f}, "
      f"lambda2={p0_w[2]:.5f} mm, n2={p0_w[3]:.3f}, q={p0_w[4]:.3f}")

weib_ok = False
try:
    popt_w, pcov_w = curve_fit(
        bi_weibull, phi_labels, mass_frac,
        p0=p0_w, bounds=(lb_w, ub_w), maxfev=30000
    )
    perr_w    = np.sqrt(np.diag(pcov_w))
    lam1_w, n1_w, lam2_w, n2_w, q_w = popt_w
    yw_fit    = bi_weibull(phi_labels, *popt_w)
    rmse_w    = np.sqrt(np.mean((mass_frac - yw_fit) ** 2)) * 100
    lam1_phi_w = -np.log2(lam1_w)
    lam2_phi_w = -np.log2(lam2_w)
    weib_ok   = True
except Exception as e:
    print(f"Bi-Weibull fit failed from clicked guess: {e}")
    print("Try re-clicking closer to each peak's true center, "
          "or widen lb_w/ub_w above.")

if weib_ok:
    print(f"\nBi-Weibull fit (from clicked starting guess):")
    print(f"  lambda1 = {lam1_w:.4f} mm ({lam1_phi_w:+.3f} Phi)  +/- {perr_w[0]:.4f}")
    print(f"  n1      = {n1_w:.4f}  +/- {perr_w[1]:.4f}")
    print(f"  lambda2 = {lam2_w:.6f} mm ({lam2_phi_w:+.3f} Phi)  +/- {perr_w[2]:.6f}")
    print(f"  n2      = {n2_w:.4f}  +/- {perr_w[3]:.4f}")
    print(f"  q       = {q_w:.4f}  +/- {perr_w[4]:.4f}")
    print(f"  RMSE    = {rmse_w:.4f} wt.%")

In [ ]:

# =================================================================
# CELL 5 — plot the fit result
# =================================================================
if not weib_ok:
    raise RuntimeError("Cannot plot: the fit in Cell 3 did not converge.")

phi_smooth = np.linspace(phi_labels.min() - 1.0, phi_labels.max() + 1.0, 1000)
CDATA, CFIT, CPOP1, CPOP2 = '#c0392b', '#2c3e50', '#2980b9', '#27ae60'

fig2 = plt.figure(figsize=(7, 5.4))
fig2.suptitle(f"{DATASET_LABEL} (click-picked starting guess)",
              fontsize=12, y=1.02, fontweight='bold')
ax2 = fig2.add_subplot(111)

ax2.bar(phi_labels, mass_pct, width=0.90, color=CDATA, alpha=0.55,
        edgecolor='grey', linewidth=0.5, label='Data')

yw_tot = bi_weibull(phi_smooth, *popt_w) * 100
yw_c   = q_w     * _weibull_comp(phi_smooth, lam1_w, n1_w) * 100
yw_f   = (1-q_w) * _weibull_comp(phi_smooth, lam2_w, n2_w) * 100

ax2.plot(phi_smooth, yw_tot, color=CFIT, lw=2.0, label='Bi-Weibull fit')
ax2.plot(phi_smooth, yw_c, color=CPOP1, lw=1.2, ls='--',
         label=f'Pop. 1  λ₁ = {lam1_w:.3f} mm')
ax2.plot(phi_smooth, yw_f, color=CPOP2, lw=1.2, ls='--',
         label=f'Pop. 2  λ₂ = {lam2_w:.5f} mm')

ax2.set_xlabel('Φ', fontsize=11)
ax2.set_ylabel('Mass (wt.%)', fontsize=11)
ax2.set_title('Bi-Weibull (click-picked)', fontsize=11, fontweight='bold')
ax2.set_xlim(phi_labels.min() - 1, phi_labels.max() + 1)
ax2.set_ylim(bottom=0)
ax2.legend(fontsize=8)
ax2.tick_params(labelsize=9)
plt.tight_layout()
plt.show()

# --- Parameter report ---
SEP = "=" * 62
print(f"\n{SEP}")
print(f"  TGSD FIT RESULTS (click-picked starting guess)")
print(f"  {DATASET_LABEL}")
print(f"{SEP}")
print("\n  BI-WEIBULL")
print(f"    lambda1 = {lam1_w:9.4f} mm  ({lam1_phi_w:+.3f} Phi)   +/- {perr_w[0]:.4f}")
print(f"    n1      = {n1_w:9.4f}             +/- {perr_w[1]:.4f}")
print(f"    lambda2 = {lam2_w:9.6f} mm  ({lam2_phi_w:+.3f} Phi)   +/- {perr_w[2]:.6f}")
print(f"    n2      = {n2_w:9.4f}             +/- {perr_w[3]:.4f}")
print(f"    q       = {q_w:9.4f}               +/- {perr_w[4]:.4f}")
print(f"    RMSE    = {rmse_w:.4f} wt.%")
print(f"\n{SEP}\n")